In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
# surpress futre warnings
import warnings
import json
from utils import * 
warnings.simplefilter(action='ignore', category=FutureWarning)
import sys
import os
from pathlib import Path
# Add the src directory to the Python path
sys.path.append(os.path.abspath("../src"))
from guess_llm.probe.test import load_model_and_config, load_data

from scipy.stats import pearsonr
import seaborn as sns
import scienceplots

sns.set_style('whitegrid')
plt.style.use('science')
plt.rcParams['font.family'] = 'sans-serif'

saves_folder = os.path.join(Path(os.getcwd()).parent, 'saves/guess_llm')

# Predicting mean, median and greedy outcome

In [ ]:
save_names = {
    "greedy" : "mag_reg_greedy/multi_func_combined_balanced_1/Llama-2-7b-hf/run_2026-03-01_08-58-13_0",
    "mean" : "mag_reg_mean/multi_func_combined_balanced_1/Llama-2-7b-hf/run_2026-03-01_08-58-14_0",
    "median" : "mag_reg_median/multi_func_combined_balanced_1/Llama-2-7b-hf/run_2026-03-01_08-58-15_0"
}
results_df = pd.DataFrame()

for target, save_path in save_names.items():
    # Load the data
    test_df = pd.read_csv(os.path.join(saves_folder, save_path, 'test_results.csv'))
    test_df['target'] = target

    save_dir_full = f'{saves_folder}/{save_path}'
    model, config = load_model_and_config(save_dir_full)

    # Load the test data
    dataloaders, meta_dfs = load_data(config, splits=['train'])
    meta_df = meta_dfs['train']

    if target == 'greedy':
        test_df['sample_average'] = np.stack(meta_df['y_greedy'].values).mean()
    elif target == 'mean':
        test_df['sample_average'] = np.stack(meta_df['y_pred'].values).mean()
    elif target == 'median':
        test_df['sample_average'] = np.median(np.stack(meta_df['y_pred'].values), axis=1).mean()

    results_df = pd.concat([results_df, test_df], ignore_index=True)

In [ ]:
import json

results_df['value'] = results_df.apply(lambda row: (
    row['sample_mean'] if row['target'] == 'mean' else
    row['sample_median'] if row['target'] == 'median' else
    row['y_greedy']
), axis=1)

results_df['pred'] = results_df['exp_pred']
results_df['mse'] = (results_df['value'] - results_df['pred'])**2

train = np.array(results_df['train'].values)
results_df['last_train_value'] = np.array([json.loads(t)[-1] for t in train])
results_df['average_train_value'] = np.array([np.mean(json.loads(t)) for t in train])

results_df['sample_average_mse'] = (results_df['sample_average'] - results_df['value'])**2
results_df['last_train_mse'] = (results_df['last_train_value'] - results_df['value'])**2
results_df['average_train_mse'] = (results_df['average_train_value'] - results_df['value'])**2

In [ ]:
results_df.groupby(['target'])[['mse', 'sample_average_mse', 'average_train_mse', 'last_train_mse']].mean()

In [ ]:
results_df['true_exponent'] = np.floor(np.log10(np.abs(results_df['value'])))
exact_match_accuracy = (results_df['pred_order'] == results_df['true_exponent']).mean()
print(f"Exact match accuracy: {exact_match_accuracy:.3f}")

within_one_accuracy = (np.abs(results_df['pred_order'] - results_df['true_exponent']) <= 1).mean()
print(f"Accuracy within ±1: {within_one_accuracy:.3f}")

# Compute prediction correctness
results_df['correct'] = results_df['pred_order'] == results_df['true_exponent']

In [ ]:
# Helper: integer magnitude (e.g., 123.4 → 3, 0.012 → -1)
def magnitude(x):
    return np.log10(np.abs(x) + 1e-8)

results_df['value_mag'] = results_df['value'].apply(magnitude)
results_df['pred_mag'] = results_df['exp_pred'].apply(magnitude)

# Plot: integer magnitude comparison
targets = ['mean', 'median', 'greedy']
fig, axes = plt.subplots(1, 4, figsize=(12, 2.4), sharex=False, sharey=False)

for ax, target in zip(axes, targets):
    data = results_df[results_df['target'] == target]
    sns.scatterplot(data=data, x='value_mag', y='pred_mag', ax=ax, alpha=0.1)
    min_val = min(data['value_mag'].min(), data['pred_mag'].min())
    max_val = max(data['value_mag'].max(), data['pred_mag'].max())
    ax.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='y = x')
    ax.set_title(f"Target: {target}")
    ax.set_ylabel("Predicted value ($\log_{10}$ scale)")
    ax.set_xlabel("True value ($\log_{10}$ scale)")
    ax.legend()

    r2, p = pearsonr(data['value_mag'], data['pred_mag'])
    ax.text(
        0.5, 0.2, 
        f'Pearson $R$: {r2:.2f}',
        transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5, edgecolor='black')
    )

# Limit to exponents of interest
exponent_range = range(-2, 4)  # inclusive range [-3, 4]
subset = results_df[results_df['true_exponent'].isin(exponent_range)]

# Group by exponent and compute accuracy
accuracy_by_exponent = (
    subset.groupby('true_exponent')['correct']
    .mean()
    .reindex(exponent_range)
    .reset_index()
)

# Plot as boxplot-like barplot
sns.barplot(data=accuracy_by_exponent, ax=axes[3], x='true_exponent', y='correct', errorbar='se')
axes[3].set_xlabel('True exponent')
axes[3].set_ylabel('Prediction Accuracy')
axes[3].set_title('Accuracy by exponent across targets')
axes[3].set_ylim(0, 1)

plt.subplots_adjust(wspace=0.5)
plt.tight_layout()
plt.show()